# 🔬 Notebook 3: Deep Dive — BFS, PYMK, and Hot Users

This notebook is where we earn our keep. We'll take three real problems and show **bad → best**:

1. **Degrees of separation** — one-sided BFS vs *bidirectional* BFS.
2. **People You May Know** — naive live scoring vs precomputed offline pipeline.
3. **The celebrity problem** — what happens when one user has 30k neighbors, and how to survive it.

All code is runnable on a small graph you can read. Then we generate a bigger power-law graph and watch the bad solutions collapse.

## 🛠️ Setup

```bash
cd 06-system-designs/linkedin-connections
uv sync
```

Then select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab has **no external services** (no Postgres, no Redis). Everything runs in-process with Python stdlib + `pydantic`, so you can focus on the concepts.

## 0. A reusable toy graph

We represent the graph as `dict[int, set[int]]` — an adjacency list. Real systems use the same shape; they just shard it.

In [ ]:
from collections import defaultdict, deque, Counter
import random, time

def build_graph(edges):
    g = defaultdict(set)
    for u, v in edges:
        if u == v: continue
        g[u].add(v); g[v].add(u)
    return g

# A hand-made graph with an obvious structure so we can eyeball answers.
small = build_graph([
    (1,2),(2,3),(3,4),(4,5),     # chain 1-2-3-4-5
    (1,6),(6,7),(7,5),           # detour 1-6-7-5
    (3,8),(8,9),(9,10),          # branch off 3
])
for u in sorted(small):
    print(f'{u}: {sorted(small[u])}')

## 1. Degrees of separation

### ❌ Bad — single-sided BFS up to depth 3

A BFS from `u` explores **all** users at distance 1, then 2, then 3. If the average user has `b` neighbors, at depth `d` we visit roughly `b^d` nodes.

For `b = 500` that's **500 → 250k → 125M** nodes. Doable for depth 2, hopeless for depth 3 unless you cap aggressively.

In [ ]:
def bfs_one_sided(g, src, dst, max_depth=3):
    if src == dst: return 0, 1
    seen = {src}
    frontier = deque([(src, 0)])
    visited = 0
    while frontier:
        node, d = frontier.popleft()
        visited += 1
        if d == max_depth: continue
        for nb in g[node]:
            if nb == dst: return d + 1, visited + 1
            if nb not in seen:
                seen.add(nb); frontier.append((nb, d + 1))
    return None, visited

print(bfs_one_sided(small, 1, 10))  # (degree, nodes_visited)

### ✅ Best — **bidirectional** BFS

Trick: BFS from **both** ends simultaneously, alternating sides, and stop as soon as the two frontiers meet.
Instead of `b^d` work, we do roughly `2·b^(d/2)` — the square root of the bad version.

For `b=500, d=4`: bad = 62 billion, good = 500k. **Five orders of magnitude faster.**

In [ ]:
def bfs_bidirectional(g, src, dst, max_depth=4):
    """Search from both ends, always expanding the SMALLER frontier, stop when they meet.

    `expanded` counts nodes whose adjacency list we had to fetch. In a sharded graph store
    that is the number of network round trips, so it — not wall-clock time in this toy —
    is the cost that actually matters.
    """
    if src == dst:
        return 0, 0

    # dist_s / dist_t: every node reached from each side, with its distance.
    dist_s, dist_t = {src: 0}, {dst: 0}
    frontier_s, frontier_t = {src}, {dst}
    expanded = 0

    while frontier_s and frontier_t:
        # Always expand the smaller side — this is what keeps the work balanced and is
        # the entire reason bidirectional search wins.
        if len(frontier_s) > len(frontier_t):
            frontier_s, frontier_t = frontier_t, frontier_s
            dist_s, dist_t = dist_t, dist_s

        if min(dist_s[n] for n in frontier_s) + min(dist_t[n] for n in frontier_t) >= max_depth:
            return None, expanded

        next_frontier = set()
        for node in frontier_s:
            expanded += 1
            for nb in g[node]:
                if nb in dist_t:                       # the two searches have met
                    return dist_s[node] + 1 + dist_t[nb], expanded
                if nb not in dist_s:                   # genuinely new to this side
                    dist_s[nb] = dist_s[node] + 1
                    next_frontier.add(nb)
        frontier_s = next_frontier
    return None, expanded

# `max_depth` is a budget, not a bug: user 10 really is 5 hops from user 1, so a
# search capped at 4 correctly refuses to answer rather than lying.
print("1 -> 10, max_depth=4 :", bfs_bidirectional(small, 1, 10, max_depth=4), " (None = 'further than 4')")
print("1 -> 10, max_depth=6 :", bfs_bidirectional(small, 1, 10, max_depth=6))
print("1 ->  5, max_depth=4 :", bfs_bidirectional(small, 1, 5,  max_depth=4))

# Correctness first: bidirectional BFS is only useful if it agrees with plain BFS.
def plain_bfs_distance(g, s, t):
    if s == t: return 0
    seen, q = {s}, deque([(s, 0)])
    while q:
        n, d = q.popleft()
        for nb in g[n]:
            if nb == t: return d + 1
            if nb not in seen:
                seen.add(nb); q.append((nb, d + 1))
    return None

pairs = [(a, b) for a in small for b in small if a < b]
assert all(bfs_bidirectional(small, a, b, max_depth=99)[0] == plain_bfs_distance(small, a, b)
           for a, b in pairs)
# ...and the depth cap must never invent an answer, nor hide one that fits the budget.
for cap in (1, 2, 3, 4, 5, 6):
    for a, b in pairs:
        got = bfs_bidirectional(small, a, b, max_depth=cap)[0]
        true = plain_bfs_distance(small, a, b)
        assert got == (true if (true is not None and true <= cap) else None), (a, b, cap, got, true)
print(f"✅ matches plain BFS on all {len(pairs)} pairs, and respects max_depth for caps 1-6")


### Benchmark on a larger synthetic graph

Let's build a preferential-attachment graph (rich get richer) — the same shape as real social networks.

In [ ]:
def preferential_attachment(n_nodes=5_000, m=5, seed=42):
    """Rich-get-richer growth: new nodes attach to existing ones with probability
    proportional to their current degree. Produces the power-law degree distribution
    that real social graphs have."""
    rng = random.Random(seed)
    g = defaultdict(set)
    # seed clique
    for i in range(m):
        for j in range(i+1, m):
            g[i].add(j); g[j].add(i)
    degree_bag = []
    for i in range(m):
        degree_bag.extend([i] * (m-1))
    for new in range(m, n_nodes):
        targets = set()
        while len(targets) < m:
            targets.add(rng.choice(degree_bag))
        for t in targets:
            g[new].add(t); g[t].add(new)
            degree_bag.extend([new, t])
    return g

big = preferential_attachment(n_nodes=5_000, m=6)
print('nodes:', len(big), '  edges:', sum(len(v) for v in big.values()) // 2)
print('max degree:', max(len(v) for v in big.values()), '(this is the "celebrity")')

# Measure NODES EXPANDED, not milliseconds. Each expansion is one adjacency-list fetch,
# i.e. one shard round trip in production — and unlike a 0.0 ms timing it is stable.
# Average over many random pairs so we are not reporting one lucky shortest path.
rng = random.Random(0)
nodes = list(big)
pairs = [tuple(rng.sample(nodes, 2)) for _ in range(200)]

tot_one = tot_bi = agree = 0
for s, d in pairs:
    d1, n1 = bfs_one_sided(big, s, d, max_depth=6)
    d2, n2 = bfs_bidirectional(big, s, d, max_depth=6)
    agree += (d1 == d2)
    tot_one += n1
    tot_bi  += n2

print(f'\nAveraged over {len(pairs)} random pairs (max_depth=6):')
print(f'  one-sided    : {tot_one/len(pairs):8.1f} nodes expanded per query')
print(f'  bidirectional: {tot_bi/len(pairs):8.1f} nodes expanded per query')
print(f'  reduction    : {tot_one/max(tot_bi,1):8.1f}x fewer adjacency fetches')
print(f'  same answer on {agree}/{len(pairs)} pairs')
print("""
Caveat worth saying out loud: 5,000 nodes is a *small world*. Almost every pair is 3 hops
apart, so one-sided BFS gives up before it can really explode. The gap you see here is the
floor, not the ceiling. The next cell projects what happens at LinkedIn's actual branching
factor, where the exponential term has room to bite.""")


In [ ]:
# Why the toy benchmark understates the win: the cost is b**d, and b is 500 in production.
b = 500   # average connections per user
print(f"{'depth':>6}{'one-sided  b^d':>24}{'bidirectional  2*b^(d/2)':>28}{'ratio':>16}")
for d in (2, 3, 4, 5, 6):
    one = b ** d
    bi  = 2 * b ** (d / 2)
    print(f"{d:>6}{one:>24,.0f}{bi:>28,.0f}{one/bi:>15,.0f}x")

print("""
At depth 4 — 'is this person within 4 hops of me?' — one-sided BFS would touch 62 billion nodes and bidirectional
would touch 500 thousand. That is the five-orders-of-magnitude claim, and it is arithmetic,
not a measurement: no laptop can run the 62-billion-node side to prove it.

The honest summary: the 5k-node benchmark above proves bidirectional BFS is CORRECT and
cheaper; this table is what proves it is NECESSARY.""")


### Production note

LinkedIn doesn't run BFS on its main graph DB per request — the query fan-out would melt any database. In practice the service:

- caches each user's **1st-degree** neighbors in a fast KV store (hot adjacency),
- computes **2nd-degree** by unioning the 1st-degree sets of the user's friends (with de-dup),
- **caps at degree 3** because the world is small: in a graph of 1B nodes, almost everyone is within 4-5 hops.

## 2. People You May Know (PYMK)

**Intuition.** If you and Carol share 12 common friends, you probably know each other.
Formally: for each non-neighbor `c` of `u`, score `c` by the count of paths of length 2 between `u` and `c`, i.e. `|N(u) ∩ N(c)|`.

### ❌ Bad — live, unweighted, per-request

Compute candidates on every page view. For a user with 500 friends and each friend with 500 friends, we touch ~250k pairs **per request**.

In [ ]:
def pymk_live_naive(g, u, top_k=5):
    direct = g[u]
    scores = Counter()
    for f in direct:
        for fof in g[f]:
            if fof == u or fof in direct: continue
            scores[fof] += 1
    return scores.most_common(top_k)

print('pymk_live(1) in small graph:', pymk_live_naive(small, 1))

t = time.perf_counter()
for _ in range(50):
    pymk_live_naive(big, 7)   # one repeat user
print(f'50 live calls on 5k-node graph: {(time.perf_counter()-t)*1000:.0f} ms')

### 🟡 Better — weight by mutual-friend quality (Adamic–Adar)

Raw mutual-friend count is misled by celebrities: if a celebrity with 30k friends is mutual with everyone, they make every suggestion look strong.

**Adamic–Adar** down-weights high-degree mutuals: a shared friend with only 5 friends carries more signal than one with 30,000.

$$\text{score}(u, c) = \sum_{m \in N(u) \cap N(c)} \frac{1}{\log |N(m)|}$$

In [ ]:
import math

def pymk_adamic_adar(g, u, top_k=5):
    direct = g[u]
    scores = defaultdict(float)
    for f in direct:
        # Guard the log. A mutual friend of degree 1 would give 1/log(1) = 1/0, and the
        # "just add epsilon" fix is worse than the bug: 1/log(1 + 1e-9) is about 1e9, so a
        # single degree-1 mutual would outrank every other signal combined. Clamping the
        # degree at 2 keeps the weight finite and monotonically decreasing in degree.
        weight = 1.0 / math.log(max(len(g[f]), 2))
        for fof in g[f]:
            if fof == u or fof in direct:
                continue
            scores[fof] += weight
    return sorted(scores.items(), key=lambda kv: -kv[1])[:top_k]

# Show the landmine explicitly on a hand-made graph containing a degree-1 mutual.
tiny = build_graph([(1, 2), (2, 3), (1, 4), (4, 5), (4, 6), (4, 7), (4, 8)])
print("weights for a shared friend, by that friend's degree:")
for deg in (1, 2, 5, 50, 30_000):
    naive = float("inf") if deg == 1 else 1.0 / math.log(deg + 1e-9)
    print(f"  degree {deg:>6}: 1/log(d+1e-9) = {naive:>18,.2f}   clamped = {1.0/math.log(max(deg,2)):.4f}")
print("  -> the epsilon version hands a degree-1 mutual ~1e9; the clamped version hands it 1.44.")

print()
print('naive    :', pymk_live_naive(big, 7, top_k=5))
print('adamic/aa:', [(n, round(s, 3)) for n, s in pymk_adamic_adar(big, 7, top_k=5)])


### ✅ Best — precompute offline, serve from KV

Real PYMK runs in a **nightly batch** (Spark / Flink). For every user we store `top_K` suggestions in a KV store keyed by user id. At read time the app just does `GET pymk:{user_id}` — O(1).

We'll simulate the offline job with a dict. In reality the 'dict' is Redis or RocksDB, and the computation is MapReduce over a billion users.

In [ ]:
def offline_pymk_job(g, top_k=5):
    out = {}
    for u in g:
        out[u] = pymk_adamic_adar(g, u, top_k)
    return out   # written to KV in production

pymk_store = offline_pymk_job(big, top_k=5)

# Serving: cheap lookup.
def serve_pymk(user_id):
    return pymk_store.get(user_id, [])

t = time.perf_counter()
for _ in range(50_000):
    serve_pymk(7)
print(f'50,000 serve_pymk calls: {(time.perf_counter()-t)*1000:.0f} ms')
print('top suggestion for user 7:', serve_pymk(7)[0])

### Why this architecture wins

| Dimension | Live naive | Offline + KV serve |
|---|---|---|
| Read latency | 10–500 ms (fan-out) | <1 ms (single KV hit) |
| Load on graph DB | 1× per page view | 0× at read time |
| Freshness | real-time | hours–day-old (fine for PYMK) |
| Cost | CPU every read | CPU once per day |

**Rule of thumb**: if a feature tolerates staleness, precompute it. If it needs to be live, cache it.

## 3. The celebrity / hot-user problem

Some users (Bill Gates, Richard Branson) have **hundreds of thousands** of connections. They create three real-world problems:

1. **Fat row**: reading their adjacency list returns megabytes of ids.
2. **Hot shard**: every query touching them hits one shard.
3. **Skewed PYMK**: every friend-of-friend path goes through them, dominating scores (that's why we used Adamic–Adar above).

Let's see the skew in our generated graph and then three mitigations.

In [ ]:
degrees = sorted((len(v) for v in big.values()), reverse=True)
print('top-10 degrees (celebrities):', degrees[:10])
print('median degree             :', degrees[len(degrees)//2])
# Notice the tail — a classic power law.

### Mitigation 1 — pagination + capped fan-out

Never return more than `N` neighbors in one API call. Cursor-paginate. Enforce a server-side max (e.g. 100).

### Mitigation 2 — cache celebrity adjacency aggressively

Celebrities' lists don't change often relative to traffic. Cache them with a **long TTL** and **background refresh**; stampede-guard with a single-flight lock.

### Mitigation 3 — exclude or down-weight high-degree mutuals in PYMK

We already did this with Adamic–Adar. An alternative: hard-skip users whose degree exceeds some threshold when computing friend-of-friend scores.

In [ ]:
def pymk_skip_celebs(g, u, top_k=5, degree_cap=500):
    direct = g[u]
    scores = Counter()
    for f in direct:
        if len(g[f]) > degree_cap:
            continue            # skip 'everyone-knows-everyone' bridges
        for fof in g[f]:
            if fof == u or fof in direct: continue
            scores[fof] += 1
    return scores.most_common(top_k)

print('without celeb filter:', pymk_live_naive(big, 7, top_k=5))
print('with    celeb filter:', pymk_skip_celebs(big, 7, top_k=5, degree_cap=200))

## 4. One more real-world wrinkle: **sharding**

When edges are split across 1,000 database shards by `owner`, **no single shard can answer 'friends of friends'** — the 500 friends live on ~500 different shards.

Production systems (Facebook TAO, LinkedIn's graph DB) solve this with:

- A **routing layer** that fans out to the right shards in parallel.
- **Cache stickiness**: every 2nd-degree query checks the cache first; shards are touched only on misses.
- **Hedged requests**: after P95 latency, re-issue to a replica and take whichever wins.

The algorithms in this notebook don't change — only *where* they run. Your Python BFS becomes a distributed BFS with the same shape.

## 5. Closing thoughts

- **BFS everywhere** — it's the Swiss Army knife of social graphs. Bidirectional BFS costs a
  few extra lines and turns `b^d` into `2*b^(d/2)`: ~36x on the toy graph above, ~10^5 at
  LinkedIn's branching factor and depth 4.
- **Precompute slow things, cache hot things.** PYMK is slow → precompute. Adjacency is hot → cache.
- **Mind the power law.** Celebrities break naive algorithms; Adamic–Adar, degree caps, and heavy caching are the three practical fixes.

You now have all the mental tools to reason about a billion-node social graph, and runnable code for every idea in this lab.